In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

In [4]:
df = pd.read_excel("ENERGY_POVERTY_CLEANED.xlsx")
df.head()

,Country Name,Time,Access to clean fuels and technologies for cooking (% of population),"Access to clean fuels and technologies for cooking, rural (% of rural population)","Access to clean fuels and technologies for cooking, urban (% of urban population)",Access to electricity (% of population),"Access to electricity, rural (% of rural population)","Access to electricity, urban (% of urban population)",Adjusted savings: energy depletion (% of GNI),Adjusted savings: natural resources depletion (% of GNI),...,Renewable electricity output (% of total electricity output),Renewable energy consumption (% of total final energy consumption),Renewable internal freshwater resources per capita (cubic meters),"Renewable internal freshwater resources, total (billion cubic meters)",Rural population,Rural population (% of total population),Rural population growth (annual %),Urban population,Urban population (% of total population),Urban population growth (annual %)
0,Angola,2022,50.0,8.6,75.9,48.5,31.011705,76.2,3.492557,8.216224,...,38.134409,59.969,4153.216769,148.00,11374345,31.919,1.216165,24260684,68.081,4.059358
1,Benin,2022,6.0,5.1,6.5,56.5,45.500000,71.1,3.492557,8.216224,...,38.134409,59.969,748.573658,10.30,6943870,50.466,1.439953,6815631,49.534,3.688459
2,Botswana,2022,66.0,25.3,86.9,75.9,25.000000,95.5,3.492557,8.216224,...,38.134409,59.969,983.650096,2.40,677704,27.776,-0.774032,1762188,72.224,2.512128
3,Burkina Faso,2022,17.2,3.0,47.8,19.5,3.400000,60.5,3.492557,8.216224,...,38.134409,71.400,555.332485,12.50,15333832,68.123,1.378344,7175206,31.877,4.327612
4,Burundi,2022,0.1,0.1,0.2,10.3,1.600000,64.0,3.492557,8.216224,...,38.134409,83.000,755.193060,10.06,11400594,85.583,2.287245,1920503,14.417,5.227534


In [5]:
# MEPI CONSTRUCTION VARIABLES 
# 13 indicators across 5 dimensions


mepi_vars = [
    # D1 — Electricity access
    'Access to electricity (% of population)',
    'Access to electricity, rural (% of rural population)',
    'Access to electricity, urban (% of urban population)',
    # D2 — Affordability / energy efficiency
    'Energy imports, net (% of energy use)',
    'Energy intensity level of primary energy (MJ/$2017 PPP GDP)',
    'GDP per unit of energy use (constant 2021 PPP $ per kg of oil equivalent)',
    # D3 — Reliability proxies
    'Electric power transmission and distribution losses (% of output)',
    'Electric power consumption (kWh per capita)',
    # D4 — Clean cooking
    'Access to clean fuels and technologies for cooking (% of population)',
    'Access to clean fuels and technologies for cooking, rural (% of rural population)',
    'Access to clean fuels and technologies for cooking, urban (% of urban population)',
    # D5 — Sustainability
    'Renewable energy consumption (% of total final energy consumption)',
    'Carbon intensity of GDP (kg CO2e per 2021 PPP $ of GDP)',
]

# Indicators where higher value = LESS energy poverty (invert during normalisation)
good_indicators = [
    'Access to electricity (% of population)',
    'Access to electricity, rural (% of rural population)',
    'Access to electricity, urban (% of urban population)',
    'Access to clean fuels and technologies for cooking (% of population)',
    'Access to clean fuels and technologies for cooking, rural (% of rural population)',
    'Access to clean fuels and technologies for cooking, urban (% of urban population)',
    'Electric power consumption (kWh per capita)',
    'Renewable energy consumption (% of total final energy consumption)',
    'GDP per unit of energy use (constant 2021 PPP $ per kg of oil equivalent)',
]

# Indicators where higher value = MORE energy poverty
bad_indicators = [
    'Energy imports, net (% of energy use)',
    'Energy intensity level of primary energy (MJ/$2017 PPP GDP)',
    'Electric power transmission and distribution losses (% of output)',
    'Carbon intensity of GDP (kg CO2e per 2021 PPP $ of GDP)',
]

# Dimension mapping for nested weighting
dimensions = {
    'D1_electricity_access': [
        'Access to electricity (% of population)',
        'Access to electricity, rural (% of rural population)',
        'Access to electricity, urban (% of urban population)',
    ],
    'D2_affordability_efficiency': [
        'GDP per unit of energy use (constant 2021 PPP $ per kg of oil equivalent)',
        'Energy imports, net (% of energy use)',
        'Energy intensity level of primary energy (MJ/$2017 PPP GDP)',
    ],
    'D3_reliability': [
        'Electric power transmission and distribution losses (% of output)',
        'Electric power consumption (kWh per capita)',
    ],
    'D4_clean_cooking': [
        'Access to clean fuels and technologies for cooking (% of population)',
        'Access to clean fuels and technologies for cooking, rural (% of rural population)',
        'Access to clean fuels and technologies for cooking, urban (% of urban population)',
    ],
    'D5_sustainability': [
        'Renewable energy consumption (% of total final energy consumption)',
        'Carbon intensity of GDP (kg CO2e per 2021 PPP $ of GDP)',
    ],
}

print(f"Total MEPI indicators : {len(mepi_vars)}")
print(f"Good indicators       : {len(good_indicators)}")
print(f"Bad indicators        : {len(bad_indicators)}")
print(f"Dimensions            : {len(dimensions)}")

Total MEPI indicators : 13
Good indicators       : 9
Bad indicators        : 4
Dimensions            : 5


In [6]:
# VALIDATE ALL MEPI VARIABLES EXIST IN DATAFRAME 
missing = [v for v in mepi_vars if v not in df.columns]
if missing:
    raise ValueError(f"Missing from df: {missing}")
else:
    print("All MEPI variables found in df")

# Check good + bad = full mepi_vars list
check = set(good_indicators + bad_indicators) == set(mepi_vars)
print(f"Good + bad covers all MEPI vars: {check}")

# Preview the MEPI subset
X = df[['Country Name', 'Time'] + mepi_vars].copy()
print(f"\nPanel shape : {X.shape}")
print(f"Countries   : {X['Country Name'].nunique()}")
print(f"Years       : {X['Time'].nunique()} ({int(X['Time'].min())}–{int(X['Time'].max())})")
display(X.head())

All MEPI variables found in df
Good + bad covers all MEPI vars: True

Panel shape : (1242, 15)
Countries   : 54
Years       : 23 (2000–2022)


,Country Name,Time,Access to electricity (% of population),"Access to electricity, rural (% of rural population)","Access to electricity, urban (% of urban population)","Energy imports, net (% of energy use)",Energy intensity level of primary energy (MJ/$2017 PPP GDP),GDP per unit of energy use (constant 2021 PPP $ per kg of oil equivalent),Electric power transmission and distribution losses (% of output),Electric power consumption (kWh per capita),Access to clean fuels and technologies for cooking (% of population),"Access to clean fuels and technologies for cooking, rural (% of rural population)","Access to clean fuels and technologies for cooking, urban (% of urban population)",Renewable energy consumption (% of total final energy consumption),Carbon intensity of GDP (kg CO2e per 2021 PPP $ of GDP)
0,Angola,2022,48.5,31.011705,76.2,-353.971819,6.327421,17.483084,11.266891,392.507047,50.0,8.6,75.9,59.969,0.103763
1,Benin,2022,56.5,45.500000,71.1,40.903170,6.327421,9.547964,38.461538,105.309052,6.0,5.1,6.5,59.969,0.134060
2,Botswana,2022,75.9,25.000000,95.5,23.681778,6.327421,16.004273,24.215071,1522.198524,66.0,25.3,86.9,59.969,0.160068
3,Burkina Faso,2022,19.5,3.400000,60.5,-129.066628,5.400000,7.880550,12.462552,131.191746,17.2,3.0,47.8,71.400,0.110684
4,Burundi,2022,10.3,1.600000,64.0,13.773127,7.400000,6.571780,45.646393,258.855277,0.1,0.1,0.2,83.000,0.075993


In [7]:
# MISSINGNESS REPORT BEFORE IMPUTATION 
missing_report = (
    X[mepi_vars]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .rename('missing_%')
    .to_frame()
)
missing_report['missing_n'] = X[mepi_vars].isna().sum().values
missing_report = missing_report.sort_values('missing_%', ascending=False)

print("Missingness per indicator:")
display(missing_report)

# Flag country-years with too many missing indicators
X['missing_count'] = X[mepi_vars].isna().sum(axis=1)
X['reliable']      = X['missing_count'] <= 2

print(f"\nTotal observations       : {len(X)}")
print(f"Reliable (≤2 missing)    : {X['reliable'].sum()}")
print(f"Unreliable (>2 missing)  : {(~X['reliable']).sum()}")

Missingness per indicator:


,missing_%,missing_n
Access to electricity (% of population),0.0,0
"Access to electricity, rural (% of rural population)",0.0,0
"Access to electricity, urban (% of urban population)",0.0,0
"Energy imports, net (% of energy use)",0.0,0
Energy intensity level of primary energy (MJ/$2017 PPP GDP),0.0,0
GDP per unit of energy use (constant 2021 PPP $ per kg of oil equivalent),0.0,0
Electric power transmission and distribution losses (% of output),0.0,0
Electric power consumption (kWh per capita),0.0,0
Access to clean fuels and technologies for cooking (% of population),0.0,0
"Access to clean fuels and technologies for cooking, rural (% of rural population)",0.0,0



Total observations       : 1242
Reliable (≤2 missing)    : 1242
Unreliable (>2 missing)  : 0


In [9]:
#  FIXED THEORETICAL BOUNDS 

fixed_bounds = {
    'Access to electricity (% of population)':                                    (0, 100),
    'Access to electricity, rural (% of rural population)':                       (0, 100),
    'Access to electricity, urban (% of urban population)':                       (0, 100),
    'Access to clean fuels and technologies for cooking (% of population)':       (0, 100),
    'Access to clean fuels and technologies for cooking, rural (% of rural population)': (0, 100),
    'Access to clean fuels and technologies for cooking, urban (% of urban population)': (0, 100),
    'Electric power consumption (kWh per capita)':                                (0, 3000),
    'Renewable energy consumption (% of total final energy consumption)':         (0, 100),
    'GDP per unit of energy use (constant 2021 PPP $ per kg of oil equivalent)':  (0, 25),
    'Electric power transmission and distribution losses (% of output)':          (0, 60),
    'Carbon intensity of GDP (kg CO2e per 2021 PPP $ of GDP)':                   (0, 2),
    'Energy imports, net (% of energy use)':                                      (-100, 100),
    'Energy intensity level of primary energy (MJ/$2017 PPP GDP)':               (0, 20),
}

# NORMALISATION 

X_norm = pd.DataFrame(index=X.index, columns=mepi_vars, dtype=float)

for col in mepi_vars:
    lo, hi   = fixed_bounds[col]
    col_data = X[col].astype(float)
    if col in good_indicators:
        X_norm[col] = (hi - col_data) / (hi - lo)
    else:
        X_norm[col] = (col_data - lo) / (hi - lo)
    X_norm[col] = X_norm[col].clip(0, 1)

# Carry ID columns
X_norm['Country Name'] = X['Country Name'].values
X_norm['Time']         = X['Time'].values

print("Normalisation complete")
print(f"Score range — min: {X_norm[mepi_vars].min().min():.3f}, "
      f"max: {X_norm[mepi_vars].max().max():.3f}")
display(X_norm[['Country Name', 'Time'] + mepi_vars].head())

Normalisation complete
Score range — min: 0.000, max: 1.000


,Country Name,Time,Access to electricity (% of population),"Access to electricity, rural (% of rural population)","Access to electricity, urban (% of urban population)","Energy imports, net (% of energy use)",Energy intensity level of primary energy (MJ/$2017 PPP GDP),GDP per unit of energy use (constant 2021 PPP $ per kg of oil equivalent),Electric power transmission and distribution losses (% of output),Electric power consumption (kWh per capita),Access to clean fuels and technologies for cooking (% of population),"Access to clean fuels and technologies for cooking, rural (% of rural population)","Access to clean fuels and technologies for cooking, urban (% of urban population)",Renewable energy consumption (% of total final energy consumption),Carbon intensity of GDP (kg CO2e per 2021 PPP $ of GDP)
0,Angola,2022,0.515,0.689883,0.238,0.000000,0.316371,0.300677,0.187782,0.869164,0.500,0.914,0.241,0.40031,0.051882
1,Benin,2022,0.435,0.545000,0.289,0.704516,0.316371,0.618081,0.641026,0.964897,0.940,0.949,0.935,0.40031,0.067030
2,Botswana,2022,0.241,0.750000,0.045,0.618409,0.316371,0.359829,0.403585,0.492600,0.340,0.747,0.131,0.40031,0.080034
3,Burkina Faso,2022,0.805,0.966000,0.395,0.000000,0.270000,0.684778,0.207709,0.956269,0.828,0.970,0.522,0.28600,0.055342
4,Burundi,2022,0.897,0.984000,0.360,0.568866,0.370000,0.737129,0.760773,0.913715,0.999,0.999,0.998,0.17000,0.037997


In [10]:
# DIMENSION SCORES 

dim_weight = 1 / len(dimensions)   # 0.20 per dimension

dim_scores = pd.DataFrame(index=X_norm.index)
dim_scores['Country Name'] = X_norm['Country Name'].values
dim_scores['Time']         = X_norm['Time'].values

for dim_name, indicators in dimensions.items():
    available = [i for i in indicators if i in X_norm.columns]
    dim_scores[dim_name] = X_norm[available].mean(axis=1)

print("Dimension scores computed")
display(dim_scores.head())

Dimension scores computed


,Country Name,Time,D1_electricity_access,D2_affordability_efficiency,D3_reliability,D4_clean_cooking,D5_sustainability
0,Angola,2022,0.480961,0.205683,0.528473,0.551667,0.226096
1,Benin,2022,0.423000,0.546323,0.802961,0.941333,0.233670
2,Botswana,2022,0.345333,0.431536,0.448093,0.406000,0.240172
3,Burkina Faso,2022,0.722000,0.318259,0.581989,0.773333,0.170671
4,Burundi,2022,0.747000,0.558665,0.837244,0.998667,0.103998


In [11]:
#ALKIRE-FOSTER DUAL CUTOFF 

# First cutoff z_j — indicator level deprivation threshold
z_j = 1/3

# Binary deprivation matrix
g = (X_norm[mepi_vars] > z_j).astype(float)

# Indicator weights (nested: dim_weight / n_indicators in dim)
indicator_weights = {}
for dim_name, indicators in dimensions.items():
    available = [i for i in indicators if i in X_norm.columns]
    for ind in available:
        indicator_weights[ind] = dim_weight / len(available)

print("Indicator weights:")
for ind, w in indicator_weights.items():
    print(f"  {ind[:65]:<65} {w:.4f}")
print(f"\nSum of weights: {sum(indicator_weights.values()):.4f}  (must = 1.0)")

# Weighted deprivation score c_i
c = pd.Series(0.0, index=X_norm.index)
for ind, w in indicator_weights.items():
    c += g[ind] * w

dim_scores['deprivation_score_c'] = c.values

# Second cutoff k — poverty identification
k = 1/3
dim_scores['is_poor']    = (c >= k).astype(int)
dim_scores['c_censored'] = np.where(dim_scores['is_poor'] == 1, c, 0)

print(f"\nPoverty cutoff k        : {k:.3f}")
print(f"Energy poor obs         : {dim_scores['is_poor'].sum()}")
print(f"Not energy poor obs     : {(dim_scores['is_poor'] == 0).sum()}")
print(f"% classified as poor    : {dim_scores['is_poor'].mean()*100:.1f}%")

Indicator weights:
  Access to electricity (% of population)                           0.0667
  Access to electricity, rural (% of rural population)              0.0667
  Access to electricity, urban (% of urban population)              0.0667
  GDP per unit of energy use (constant 2021 PPP $ per kg of oil equ 0.0667
  Energy imports, net (% of energy use)                             0.0667
  Energy intensity level of primary energy (MJ/$2017 PPP GDP)       0.0667
  Electric power transmission and distribution losses (% of output) 0.1000
  Electric power consumption (kWh per capita)                       0.1000
  Access to clean fuels and technologies for cooking (% of populati 0.0667
  Access to clean fuels and technologies for cooking, rural (% of r 0.0667
  Access to clean fuels and technologies for cooking, urban (% of u 0.0667
  Renewable energy consumption (% of total final energy consumption 0.1000
  Carbon intensity of GDP (kg CO2e per 2021 PPP $ of GDP)           0.1000

Sum o

In [12]:
# MEPI = H × A 

dim_cols = list(dimensions.keys())
dim_scores['MEPI'] = dim_scores[dim_cols].mul(dim_weight).sum(axis=1)

# Country-level summary
def intensity(x):
    poor = x[x > 0]
    return poor.mean() if len(poor) > 0 else 0

summary = (
    dim_scores
    .groupby('Country Name')
    .agg(
        H         = ('is_poor',    'mean'),
        A         = ('c_censored', intensity),
        MEPI_mean = ('MEPI',       'mean'),
        n_obs     = ('Time',       'count'),
    )
    .assign(MEPI_AF = lambda d: d['H'] * d['A'])
    .round(4)
    .sort_values('MEPI_AF', ascending=False)
)

print("Country-level MEPI summary (top 15):")
display(summary.head(15))

Country-level MEPI summary (top 15):


,H,A,MEPI_mean,n_obs,MEPI_AF
Country Name,,,,,
Benin,1.0,0.7986,0.6420,23,0.7986
Lesotho,1.0,0.7986,0.5764,23,0.7986
Burundi,1.0,0.7652,0.6157,23,0.7652
Central African Republic,1.0,0.7609,0.6319,23,0.7609
Uganda,1.0,0.7594,0.6222,23,0.7594
Djibouti,1.0,0.7580,0.5851,23,0.7580
Togo,1.0,0.7551,0.6246,23,0.7551
Sierra Leone,1.0,0.7449,0.6292,23,0.7449
Guinea-Bissau,1.0,0.7319,0.6047,23,0.7319


In [13]:
# ── DIMENSION DECOMPOSITION ───────────────────────────────────────────────────

poor_mask = dim_scores['is_poor'] == 1

decomp = {}
for dim_name in dim_cols:
    H_d = (dim_scores.loc[poor_mask, dim_name] > z_j).mean()
    decomp[dim_name] = {
        'Deprivation rate (H_d)'    : round(H_d, 4),
        'Contribution to MEPI (%)' : round(
            (dim_weight * H_d) / dim_scores['MEPI'].mean() * 100, 2)
    }

decomp_df = pd.DataFrame(decomp).T
decomp_df.index = [
    'Electricity access',
    'Affordability & efficiency',
    'Reliability',
    'Clean cooking',
    'Sustainability'
]

print("Dimension decomposition:")
display(decomp_df)

Dimension decomposition:


,Deprivation rate (H_d),Contribution to MEPI (%)
Electricity access,0.8242,32.39
Affordability & efficiency,0.7380,29.00
Reliability,0.9586,37.67
Clean cooking,0.8943,35.15
Sustainability,0.2477,9.73


In [17]:
#  DRIVER VARIABLE DEFINITIONS

#  Economic & Income drivers
economic_drivers = [
    'GDP growth (annual %)',
    'GDP (current US$)',
    'Final consumption expenditure (% of GDP)',
    'Inflation, consumer prices (annual %)',
    'External debt stocks (% of GNI)',
    'Debt service (PPG and IMF only, % of exports of goods, services and primary income)',
    'Net primary income (BoP, current US$)',
    'Fuel exports (% of merchandise exports)',
    'Fuel imports (% of merchandise imports)',
]

#Natural resource & Fiscal drivers
resource_drivers = [
    'Oil rents (% of GDP)',
    'Natural gas rents (% of GDP)',
    'Adjusted savings: energy depletion (% of GNI)',
    'Adjusted savings: natural resources depletion (% of GNI)',
    'Adjusted savings: particulate emission damage (% of GNI)',
]

#Institutional & Governance drivers
governance_drivers = [
    'Government Effectiveness: Estimate'
    
]

#Demographic & Spatial drivers
demographic_drivers = [
    'Population, total',
    'Population growth (annual %)',
    'Population density (people per sq. km of land area)',
    'Rural population (% of total population)',
    'Rural population growth (annual %)',
    'Urban population (% of total population)',
    'Urban population growth (annual %)',
]

# Environmental drivers
environmental_drivers = [
    'Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)',
    'Carbon dioxide (CO2) emissions (total) excluding LULUCF (% change from 1990)',
    'Renewable internal freshwater resources per capita (cubic meters)',
]

#Structural & Energy mix drivers
structural_drivers = [
    'Fossil fuel energy consumption (% of total)',
    'Alternative and nuclear energy (% of total energy use)',
    'Electricity production from coal sources (% of total)',
    'Electricity production from hydroelectric sources (% of total)',
    'Electricity production from natural gas sources (% of total)',
    'Electricity production from oil sources (% of total)',
    'Electricity production from nuclear sources (% of total)',
    'Electricity production from renewable sources, excluding hydroelectric (% of total)',
    'Energy use (kg of oil equivalent per capita)',
]

# Combined
driver_vars = (
    economic_drivers   +
    resource_drivers   +
    governance_drivers +
    demographic_drivers +
    environmental_drivers +
    structural_drivers
)

print(f"Total driver variables : {len(driver_vars)}")

Total driver variables : 34


In [18]:
# MERGE MEPI WITH DRIVER VARIABLES 

driver_vars = (
    economic_drivers   +
    resource_drivers   +
    governance_drivers +
    demographic_drivers +
    environmental_drivers +
    structural_drivers
)

# MEPI country-year scores
mepi_output = dim_scores[[
    'Country Name', 'Time',
    'D1_electricity_access',
    'D2_affordability_efficiency',
    'D3_reliability',
    'D4_clean_cooking',
    'D5_sustainability',
    'deprivation_score_c',
    'is_poor',
    'MEPI'
]].copy()

# Driver variables from original df
drivers_df = df[
    ['Country Name', 'Time'] +
    [v for v in driver_vars if v in df.columns]
].copy()

# Merge on Country Name + Time
final_df = pd.merge(
    mepi_output,
    drivers_df,
    on  = ['Country Name', 'Time'],
    how = 'inner'
)

print(f"Final merged dataset shape : {final_df.shape}")
print(f"Countries                  : {final_df['Country Name'].nunique()}")
print(f"Years                      : {final_df['Time'].nunique()}")
print(f"MEPI score range           : {final_df['MEPI'].min():.3f} – "
      f"{final_df['MEPI'].max():.3f}")
display(final_df.head())

# Save outputs
final_df.to_csv('mepi_with_drivers.csv', index=False)
summary.to_csv('mepi_country_summary.csv')
print("\nSaved: mepi_with_drivers.csv")
print("Saved: mepi_country_summary.csv")

Final merged dataset shape : (1242, 44)
Countries                  : 54
Years                      : 23
MEPI score range           : 0.126 – 0.700


,Country Name,Time,D1_electricity_access,D2_affordability_efficiency,D3_reliability,D4_clean_cooking,D5_sustainability,deprivation_score_c,is_poor,MEPI,...,Renewable internal freshwater resources per capita (cubic meters),Fossil fuel energy consumption (% of total),Alternative and nuclear energy (% of total energy use),Electricity production from coal sources (% of total),Electricity production from hydroelectric sources (% of total),Electricity production from natural gas sources (% of total),Electricity production from oil sources (% of total),Electricity production from nuclear sources (% of total),"Electricity production from renewable sources, excluding hydroelectric (% of total)",Energy use (kg of oil equivalent per capita)
0,Angola,2022,0.480961,0.205683,0.528473,0.551667,0.226096,0.466667,1,0.398576,...,4153.216769,0.000000,6.55,8.033157,70.424412,10.892597,16.221531,0.098538,3.805161,423.122518
1,Benin,2022,0.423000,0.546323,0.802961,0.941333,0.233670,0.766667,1,0.589457,...,748.573658,0.000000,0.05,8.033157,38.094979,73.026973,23.576424,0.098538,3.805161,375.820984
2,Botswana,2022,0.345333,0.431536,0.448093,0.406000,0.240172,0.633333,1,0.374227,...,983.650096,0.000000,0.08,96.193093,38.094979,15.114333,3.571429,0.098538,3.805161,1165.149687
3,Burkina Faso,2022,0.722000,0.318259,0.581989,0.773333,0.170671,0.566667,1,0.513251,...,555.332485,0.000000,0.18,8.033157,4.913122,15.114333,85.979629,0.098538,3.805161,312.929086
4,Burundi,2022,0.747000,0.558665,0.837244,0.998667,0.103998,0.800000,1,0.649115,...,755.193060,11.120924,0.48,8.033157,38.094979,15.114333,37.074181,0.098538,3.805161,447.479810



Saved: mepi_with_drivers.csv
Saved: mepi_country_summary.csv
